### Text Preprocessing

#### import libraries

In [40]:
import pandas as pd
import numpy as np
import re

from nltk.corpus import stopwords
import nltk

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [41]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aximsoft/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load the datasets

In [42]:
data=pd.read_csv('../dataset/IMDB Dataset.csv')

### Missing Vlaues

In [43]:
print(data.isnull().sum())

review       0
sentiment    0
dtype: int64


##### No missing values in the datasets

### Handle Duplicates

In [44]:
print("Duplicate rows:", data.duplicated().sum())

Duplicate rows: 418


In [45]:
duplicates=data[data['review'].duplicated(keep=False)]
duplicates.head(20)

,review,sentiment
42,"Of all the films I have seen, this one, The Ra...",negative
84,"We brought this film as a joke for a friend, a...",negative
140,"Before I begin, let me get something off my ch...",negative
219,Ed Wood rides again. The fact that this movie ...,negative
245,I have seen this film at least 100 times and I...,positive
480,From director Barbet Schroder (Reversal of For...,negative
513,"The story and the show were good, but it was r...",negative
636,I rented this thinking it would be pretty good...,negative
638,This movie has everything typical horror movie...,positive
701,I Enjoyed Watching This Well Acted Movie Very ...,positive


In [46]:
duplicates["sentiment"].value_counts()

sentiment
negative    598
positive    226
Name: count, dtype: int64

In [47]:
print("Before removing duplicates:")
print(data["sentiment"].value_counts())

print(
    data["sentiment"].value_counts(normalize=True) * 100
)

Before removing duplicates:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64


In [48]:
df_no_duplicates = data.drop_duplicates()

In [49]:
print("After removing duplicates:")
print(df_no_duplicates["sentiment"].value_counts())

print(
    df_no_duplicates["sentiment"].value_counts(normalize=True) * 100
)

After removing duplicates:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64
sentiment
positive    50.187568
negative    49.812432
Name: proportion, dtype: float64


In [50]:
data = data.drop_duplicates(subset=["review"]).copy()

In [51]:
print("Dataset shape:", data.shape)
print("Duplicates:", data["review"].duplicated().sum())

Dataset shape: (49582, 2)
Duplicates: 0


### Convert sentiment into numbers

In [52]:
data["label"] = data["sentiment"].map({"negative": 0,"positive":1})

In [53]:
data[["sentiment", "label"]].head()

,sentiment,label
0,positive,1
1,positive,1
2,positive,1
3,negative,0
4,positive,1


### Create the text preprocessing function

In [54]:
# Get English stopwords
stop_words = set(stopwords.words("english"))
# Keep important negation words for sentiment analysis
negation_words =  {
    "no",
    "not",
    "nor",
    "never",
    "neither",
    "none",
    "nobody",
    "nothing",
    "nowhere",
    "hardly",
    "scarcely",
    "barely",
    "isn't",
    "wasn't",
    "didn't",
    "don't",
    "can't",
    "couldn't",
    "won't"
}
# Remove negation words from the stopword list
stop_words = stop_words - negation_words

In [55]:
def preprocess_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # 3. Remove punctuation and unnecessary characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    # 4. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    # 5. Remove stopwords
    #    but keep: not, no, nor, never
    words = text.split()
    words = [
        word for word in words
        if word not in stop_words
    ]
    # 6. Join words back together
    text = " ".join(words)
    return text

### Test the preprocessing function

In [56]:
sample_review = data["review"].iloc[0]
print("Original:")
print(sample_review)
print("\nPreprocessed:")
print(preprocess_text(sample_review))

Original:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due 

### Apply preprocessing to all reviews

In [57]:
data["cleaned_review"] = data["review"].apply(preprocess_text)

In [58]:
data[["review","cleaned_review"]].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [59]:
data.head()

,review,sentiment,label,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,1,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,positive,1,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,1,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,0,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,petter mattei love time money visually stunnin...


#### Seperate X and y

In [60]:
X = data["cleaned_review"]
y = data["label"]

#### Create Train, Validation and Test datasets

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)

In [62]:
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train,test_size=0.1765,random_state=42,stratify=y_train)

In [63]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 34705
Validation: 7439
Testing: 7438


### Tokenization

In [64]:
vocab_size = 20000
tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")

In [65]:
tokenizer.fit_on_texts(X_train)

### Convert text into sequences

In [66]:
#Training
X_train_sequences = tokenizer.texts_to_sequences(X_train)
#Validation
X_val_sequences = tokenizer.texts_to_sequences(X_val)
#Testing
X_test_sequences = tokenizer.texts_to_sequences(X_test)

In [67]:
print(X_train.iloc[0])
print(X_train_sequences[0])

not much say one except probably worst early spate zombie movies may get watch another one revolt zombies month star john carradine intention building army service third reich not seen much james baskett uncle remus song south plays leader also serves carradine manservant black comic mantan moreland reprises fraidy cat chauffeur role king zombies exotically named madame sul te wan carradine housekeeper unfortunately carradine supreme achievement zombification wife brings sorts trouble not relatives turn remote abode lab inquire sudden death means fake funeral service actually proves disobedient indignant eventually persuading fellow zombies rise master also involved cowboy star bob steele still best known bit howard hawks big sleep plays u secret agent posing nazi posing sheriff thankfully director sekely would much better luck next genre effort day triffids
[4, 16, 51, 5, 432, 130, 144, 291, 1, 826, 27, 96, 18, 31, 64, 5, 7914, 1119, 3151, 206, 193, 4363, 3441, 1181, 1078, 2168, 743, 

### Padding

In [68]:
max_length = 200

In [69]:
#Train data
X_train_padded = pad_sequences(X_train_sequences,maxlen=max_length,padding="post",truncating="post")
#Validation
X_val_padded = pad_sequences(X_val_sequences,maxlen=max_length,padding="post",truncating="post")
#Test
X_test_padded = pad_sequences(X_test_sequences,maxlen=max_length,padding="post",truncating="post")

In [70]:
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)
print("Testing shape:", X_test_padded.shape)

Training shape: (34705, 200)
Validation shape: (7439, 200)
Testing shape: (7438, 200)


### Check a padded sequence

In [71]:
print(X_train_padded[0])

[    4    16    51     5   432   130   144   291     1   826    27    96
    18    31    64     5  7914  1119  3151   206   193  4363  3441  1181
  1078  2168   743 18546     4    34    16   474     1  1624     1   441
  1110   187  1616    21  2389  4363     1   204   575     1     1 14458
     1  1031 13719   107   552  1119     1   634 12228     1 12457  7291
  4363  9450   351  4363  5739  3343     1   203   815  2422   932     4
  4582   342  2757     1  3521     1  2028   213   686   994  3686  2168
    66  1486     1     1   716     1  1477  1119  2045   983    21   454
  2722   206  1678  7819    50    41   416   120  1591 11470    89  1503
   187   971   897  1294  6254  2455  6254  1961  2288    62     1    11
    16    47  1882   245   374   635   142     1     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

In [72]:
tokenizer.word_index

{'<OOV>': 1,
 'movie': 2,
 'film': 3,
 'not': 4,
 'one': 5,
 'like': 6,
 'good': 7,
 'no': 8,
 'time': 9,
 'even': 10,
 'would': 11,
 'story': 12,
 'really': 13,
 'see': 14,
 'well': 15,
 'much': 16,
 'bad': 17,
 'get': 18,
 'great': 19,
 'people': 20,
 'also': 21,
 'first': 22,
 'made': 23,
 'make': 24,
 'could': 25,
 'way': 26,
 'movies': 27,
 'think': 28,
 'characters': 29,
 'character': 30,
 'watch': 31,
 'films': 32,
 'two': 33,
 'seen': 34,
 'many': 35,
 'love': 36,
 'acting': 37,
 'plot': 38,
 'never': 39,
 'life': 40,
 'best': 41,
 'show': 42,
 'know': 43,
 'little': 44,
 'ever': 45,
 'man': 46,
 'better': 47,
 'end': 48,
 'scene': 49,
 'still': 50,
 'say': 51,
 'scenes': 52,
 'something': 53,
 'go': 54,
 'back': 55,
 'real': 56,
 'thing': 57,
 'watching': 58,
 'actors': 59,
 'years': 60,
 'though': 61,
 'director': 62,
 'funny': 63,
 'another': 64,
 'old': 65,
 'actually': 66,
 'work': 67,
 'makes': 68,
 'nothing': 69,
 'look': 70,
 'going': 71,
 'find': 72,
 'lot': 73,
 'new'

### Final labels

In [73]:
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

In [74]:
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(34705,)
(7439,)
(7438,)


In [75]:
import pickle
with open("../models/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

In [76]:
import numpy as np
np.save("../models/X_train_padded.npy", X_train_padded)
np.save("../models/X_val_padded.npy", X_val_padded)
np.save("../models/X_test_padded.npy", X_test_padded)
np.save("../models/y_train.npy", y_train)
np.save("../models/y_val.npy", y_val)
np.save("../models/y_test.npy", y_test)

In [77]:
X_train.to_pickle("../models/X_train_text.pkl")
X_val.to_pickle("../models/X_val_text.pkl")
X_test.to_pickle("../models/X_test_text.pkl")